# Set Up

**Token-augmented dual autoencoder.** Cell-level only: a transcriptomics branch (`trans_enc`/`trans_dec`) and a cell-level histology branch built from ViT *token* embeddings (16x16 sub-tiles per 224px patch, `token_enc`/`token_dec`). The patch-level (hist<->trans) branch has been removed from the project entirely -- see `functions/dual_autoencoder.py`.

**Before running this notebook**, per-slide token data must exist at `data/{slide}/01_preprocessing/unimodel/metadata/cell_token_embeddings.h5ad` -- produced by `token_preprocessing.py`:
```
python token_preprocessing.py --slide CRC_P1
```
This distills the raw ~9GB/slide token grid down to two compact per-cell vectors (`footprint`: tokens under the nucleus; `halo`: tokens in a small ring around it), so the notebook itself never touches the raw per-token files.

In [1]:
# import
import sys

import anndata as ad
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import scanpy as sc
import scipy.sparse as sp
import squidpy as sq
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
from torch_geometric.nn import TransformerConv

%load_ext autoreload
%autoreload 2

# Finds the ancestor directory that actually contains `functions/` -- the repo
# root (this notebook sits two levels under it).
for _candidate in (Path(".").resolve(), *Path(".").resolve().parents):
    if (_candidate / "functions").is_dir():
        sys.path.insert(0, str(_candidate))
        break

from functions.dual_autoencoder import *

/home/charlottewolf/miniconda3/envs/autoencoder/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
CONFIG = dict(
    sample="combined",       # str -> single slide. list[str] or "combined" -> multi-slide.
    bin_size="002um",
    dir_cellseg="cell_seg_stardist",  # in case of multiple cell segmentation methods
    loss_fct="ntxent",                # optional: ntxent, cosine

    # model
    latent_dim=64,                      # latent dim for encoders
    proj_dim=32,                        # dimension of projection head for contrastive learning
    encoder="transconv",                # sage or transconv for transcriptomics encoding

    # graph / batching
    n_neighs=6,                         # how many neighbors in knn spatial graph of cells
    num_neighbors=(10, 10),
    patches_per_batch=64,
    max_cells_per_patch=64,

    # loss
    temperature=0.07,
    neighbor_radius=2912,     # still required by compute_patch_metadata/build_pipeline_data's
                                # shared patch-batching path (2x patch diameter, 224x224px)

    # false-negative protection: excludes transcriptomically near-duplicate,
    # non-matching cells from hard negatives (see patch_level_infonce). None disables.
    # None (was 0.9): the 18-config sweep showed this masking doesn't clearly help
    # (None beat 0.9 in most pairs, e.g. weight=1.0/warmup=10: cell_top1 0.0663 vs
    # 0.0579) and it was the collapse mechanism (checks the model's own live
    # projections, so drifting projections get masked out of the negative set).
    local_anchor_sim_threshold=None,

    weight_reconstruction_by_patch_freq=True,

    # training
    n_epochs=100,
    lr=1e-3,
    warmup_epochs=5,

    # --- token branch ---
    token_pretrain_epochs=100,
    token_pretrain_lr=1e-3,
    token_pretrain_weight_decay=1e-5,
    token_pretrain_val_frac=0.1,
    token_pretrain_patience=15,
    # token_enc/token_dec are already well-pretrained (Phase 1) -- reduced LR so
    # the joint objective fine-tunes gently rather than overwriting that structure
    # at the same rate as trans_enc (training from scratch this phase).
    token_lr_mult=0.3,
    # False (was True): goal is predicting trans embeddings from histology alone,
    # which needs token_enc's actual output to move toward trans_enc's embedding
    # space, not just the narrow contrastive projection heads. freeze_token=True
    # previously blocked all Phase-2 gradient to token_enc regardless of this flag
    # (a bug, now fixed) -- token_dec stays attached so loss_recon_token keeps
    # regularizing against collapse while token_enc moves.
    freeze_token=False,

    # median nearest-cell spacing ~27px. Was 50 (~1.85x spacing) -- too small,
    # produced a checkerboard-like cluster artifact from near cells still being
    # treated as hard negatives. 100 (~3.7x) is a first attempt at more room.
    local_neighbor_radius=100,

    # History: 1.0/5-epoch-warmup collapsed (contrast_local -> 0 for ~90 epochs,
    # top-1=0 on all 7 slides) -- unfreezing token_enc while ramping to full
    # strength in 5 epochs let its projections drift together fast, and
    # local_anchor_sim_threshold's masking is circular (checks the model's own
    # live projections), so drifting pairs got masked out of the negative set,
    # a self-reinforcing loop. Reverted to 0.5/20, stable (cell_top1=0.088). The
    # 18-config sweep then found the weight itself wasn't the danger -- every
    # warmup>=10 config stayed stable, and higher weight + shorter warmup both
    # improved retrieval. Best of 18: weight=1.0/warmup=10 -> cell_top1=0.0663,
    # cluster_top1=0.5015 (vs. 0.0532/0.4923 at 0.5/20).
    max_local_contrast_weight=1.0,
    local_warmup_epochs=10,
)

In [6]:
basedir = Path("../../data")
dir_histology = basedir / "combined/01_preprocessing/unimodel/metadata"
dir_transcriptomics = basedir / "combined/01_preprocessing" / CONFIG["dir_cellseg"] / CONFIG["bin_size"] / "custom_binning"
_run_tag = CONFIG["sample"] if isinstance(CONFIG["sample"], str) else "combined"
dir_output = basedir / _run_tag / "03_integration/autoencoders" / CONFIG["bin_size"] / "with_tokens"
dir_output.mkdir(parents=True, exist_ok=True)
print("Writing outputs to:", dir_output)

Writing outputs to: ../../data/combined/03_integration/autoencoders/002um/with_tokens


# Running the Pipeline

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -- load transcriptomics + patch histology (histology only for cell<->patch pairing) --
f_hist_adata = sc.read_h5ad(dir_histology / "filtered_WSI_patch_embeddings.h5ad")
f_trans_adata = ad.read_h5ad(dir_transcriptomics / "count_area_filtered_adata.h5ad")
f_trans_coord = gpd.read_parquet(dir_transcriptomics / "nuclei_geometry.parquet")

if CONFIG["sample"] == "combined":
    pass
elif isinstance(CONFIG["sample"], str):
    f_hist_adata = f_hist_adata[f_hist_adata.obs["Slide_ID"] == CONFIG["sample"]]
    f_trans_adata = f_trans_adata[f_trans_adata.obs["Slide_ID"] == CONFIG["sample"]]
    f_trans_coord = f_trans_coord[f_trans_coord["Slide_ID"] == CONFIG["sample"]]
elif isinstance(CONFIG["sample"], (list, tuple)):
    f_hist_adata = f_hist_adata[f_hist_adata.obs["Slide_ID"].isin(CONFIG["sample"])]
    f_trans_adata = f_trans_adata[f_trans_adata.obs["Slide_ID"].isin(CONFIG["sample"])]
    f_trans_coord = f_trans_coord[f_trans_coord["Slide_ID"].isin(CONFIG["sample"])]
print("Training on slides:", sorted(f_trans_adata.obs["Slide_ID"].unique().tolist()))

trans_global_id = f_trans_adata.obs["Slide_ID"].astype(str) + "_" + f_trans_adata.obs["id"].astype(str)
if not trans_global_id.is_unique:
    raise ValueError("Cannot set Slide_ID+id as index: contains duplicates")
f_trans_adata.obs = f_trans_adata.obs.set_index(trans_global_id)

hist_global_id = f_hist_adata.obs["Slide_ID"].astype(str) + "_" + f_hist_adata.obs["Patch_ID"].astype(str)
if not hist_global_id.is_unique:
    raise ValueError("Cannot set Slide_ID+Patch_ID as index: contains duplicates")
f_hist_adata.obs = f_hist_adata.obs.set_index(hist_global_id)


/home/charlottewolf/miniconda3/envs/autoencoder/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Training on slides: ['CRC_P1', 'CRC_P2', 'CRC_P5', 'Normal_P3', 'Normal_P5']


### Load token data

Loads each in-scope slide's `cell_token_embeddings.h5ad` (from `token_preprocessing.py`), concatenates them, and -- if some in-scope slides don't have token data yet -- restricts training to the slides that do (rather than silently zero-filling, which would let the model learn "zero vector = slide X" as a shortcut).

In [ ]:
slides_needed = sorted(f_trans_adata.obs["Slide_ID"].astype(str).unique())
token_adatas, slides_with_tokens = [], []
for s in slides_needed:
    p = basedir / s / "01_preprocessing/unimodel/metadata/cell_token_embeddings.h5ad"
    if p.exists():
        token_adatas.append(sc.read_h5ad(p))
        slides_with_tokens.append(s)
    else:
        print(f"WARNING: no token data for slide {s} ({p}) -- run token_preprocessing.py for it first")

missing = sorted(set(slides_needed) - set(slides_with_tokens))
if missing:
    print(f"Restricting to slides with token data: {slides_with_tokens} (dropping {missing})")
    f_hist_adata = f_hist_adata[f_hist_adata.obs["Slide_ID"].isin(slides_with_tokens)].copy()
    f_trans_adata = f_trans_adata[f_trans_adata.obs["Slide_ID"].astype(str).isin(slides_with_tokens)].copy()
    f_trans_coord = f_trans_coord[f_trans_coord["Slide_ID"].isin(slides_with_tokens)].copy()
if not token_adatas:
    raise RuntimeError("no in-scope slide has token data -- run token_preprocessing.py first")

f_token_adata = ad.concat(token_adatas, join="outer")
token_key = f_token_adata.obs["Slide_ID"].astype(str) + "_" + f_token_adata.obs["id"].astype(str)
if not token_key.is_unique:
    raise ValueError("Cannot set Slide_ID+id as token index: contains duplicates")
f_token_adata.obs = f_token_adata.obs.set_index(token_key)
f_token_adata = f_token_adata[f_token_adata.obs["matched"].astype(bool)].copy()
print(f"Loaded token data: {f_token_adata.n_obs} cells across {len(slides_with_tokens)} slide(s)")

### Data prep

SVG mask, spatial graph, cell<->patch pairing (still needed: this is how cells are grouped into patches for PatchGroupedLoader's batching), standardization, patch metadata, PyG data + loader -- plus aligning each surviving cell to its token footprint/halo vectors and building the cell-level metadata (`cell_meta`) the contrastive term needs.

In [ ]:
# 1. SVG mask (reduced gene expression vector with only SVGs kept)
print("1. SVG mask")
svg_mask = get_svg_union_mask(f_trans_adata)
print(f"{svg_mask.sum()} SVGs selected (union)")

# 2. spatial graph - build spatial graph: nodes = cells, contain gene exp vector
print("2. building spatial graph")
conn = build_spatial_graph(f_trans_adata, library_key="Slide_ID", n_neighs=CONFIG["n_neighs"], delaunay=False)

# 3. pairing -> get pairing of uni hist embedding to cell SVGs (still needed: this is
# how cells are grouped into patches for PatchGroupedLoader's batching)
print("3. pairing")
keep_mask, patch_row_trans, hist_emb = build_pairing(f_trans_adata, f_hist_adata)

# 3b. token pairing: further restrict keep_mask to cells that also have token data
# Index.isin() already returns a plain ndarray, not a Series -- no .values needed
has_token = f_trans_adata.obs_names.isin(f_token_adata.obs_names)
n_dropped = int(keep_mask.sum() - (keep_mask & has_token).sum())
if n_dropped:
    print(f"  dropping {n_dropped} cells with a hist-patch match but no token match")
keep_mask = keep_mask & has_token

trans_sub = f_trans_adata[keep_mask].copy()
conn_sub = conn[keep_mask][:, keep_mask]
X_svg = trans_sub.X[:, svg_mask]
X_svg = X_svg.toarray() if sp.issparse(X_svg) else np.asarray(X_svg)

slide_codes, slide_uniques = pd.factorize(trans_sub.obs["Slide_ID"])
n_slides = len(slide_uniques)

# 4. standardize
print("4. standardising")
X_svg, hist_emb, svg_scaler, hist_scaler = standardize_modalities(X_svg, hist_emb)

token_sub = f_token_adata[trans_sub.obs_names]
assert (token_sub.obs_names == trans_sub.obs_names).all(), "token/trans row alignment mismatch"
# footprint + halo, concatenated -- kept as two distinct signals rather than
# blended into one average (see token_preprocessing.py docstring).
token_emb = np.concatenate(
    [np.asarray(token_sub.obsm["X_token_footprint"]), np.asarray(token_sub.obsm["X_token_halo"])],
    axis=1,
).astype(np.float32)
token_scaler = StandardScaler()
token_emb = token_scaler.fit_transform(token_emb).astype(np.float32)
print(f"  token_emb: {token_emb.shape}")

# 5. patch metadata: centroid, cell count, inverse-frequency weight
print("5. centroids")
cell_xy = extract_cell_xy(trans_sub, f_trans_coord)
patch_meta, patch_meta_df = compute_patch_metadata(
    patch_row_trans, cell_xy, f_hist_adata,
    neighbor_radius=CONFIG["neighbor_radius"],
)
patch_meta_df.to_csv(dir_output / "patch_metadata.csv", index=False)

cell_meta = build_cell_meta(cell_xy, slide_codes, neighbor_radius=CONFIG["local_neighbor_radius"])

# 6. PyG data + patch-grouped loader
print("6. pyg")
data = build_pyg_data(X_svg, conn_sub, patch_row_trans, slide_codes)
patch_to_nodes = build_patch_to_nodes(data.patch_row)
loader = PatchGroupedLoader(
    data, patch_to_nodes, patch_meta["slide_id"],
    num_neighbors=CONFIG["num_neighbors"],
    patches_per_batch=CONFIG["patches_per_batch"],
    max_cells_per_patch=CONFIG["max_cells_per_patch"],
)

### Model

In [ ]:
model = DualAutoencoder(
    n_svg=X_svg.shape[1],
    latent_dim=CONFIG["latent_dim"], proj_dim=CONFIG["proj_dim"],
    n_slides=n_slides,
    encoder_type=CONFIG["encoder"],
    token_dim=token_emb.shape[1],
)
model.to(device)
token_emb_t = torch.tensor(token_emb, dtype=torch.float32)

### Phase 1 -- pretrain token branch

In [ ]:
pretrain_token_autoencoder(
    model, token_emb_t, cell_meta,
    epochs=CONFIG["token_pretrain_epochs"], lr=CONFIG["token_pretrain_lr"],
    weight_decay=CONFIG["token_pretrain_weight_decay"],
    val_frac=CONFIG["token_pretrain_val_frac"], patience=CONFIG["token_pretrain_patience"],
    device=device,
)

### Phase 2 -- freeze token branch

Gated on `CONFIG["freeze_token"]` -- previously called unconditionally here,
which meant `freeze_token`/`max_local_contrast_weight`/`token_lr_mult` had no
way to actually let `token_enc` adapt during Phase 3 no matter how they were
set (downstream goal is a trans-predictive token embedding, which needs
`token_enc`'s actual output to move, not just the projection heads used inside
the contrastive loss).

In [ ]:
if CONFIG["freeze_token"]:
    freeze_token_encoder(model, freeze_decoder=True)

### Phase 3 -- joint training (cell-level contrastive)

In [ ]:
train(
    model, loader, patch_meta,
    n_epochs=CONFIG["n_epochs"], lr=CONFIG["lr"], warmup_epochs=CONFIG["warmup_epochs"],
    weight_reconstruction=CONFIG["weight_reconstruction_by_patch_freq"],
    device=device,
    token_emb_all=token_emb_t, cell_meta=cell_meta,
    max_local_contrast_weight=CONFIG["max_local_contrast_weight"],
    local_warmup_epochs=CONFIG["local_warmup_epochs"],
    freeze_token=CONFIG["freeze_token"],
    token_lr_mult=CONFIG["token_lr_mult"],
    local_anchor_sim_threshold=CONFIG["local_anchor_sim_threshold"],
)

### Save checkpoint (before eval)

Saved immediately after training, before the eval cell below -- so a crash in `retrieval_accuracy` (e.g. an OOM building the full-cohort similarity matrix) can never lose a completed training run. The eval is diagnostic, not load-bearing.

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "epoch": CONFIG["n_epochs"] - 1,
    "n_svg": X_svg.shape[1],
    "latent_dim": CONFIG["latent_dim"],
    "proj_dim": CONFIG["proj_dim"],
    "n_slides": n_slides,
    "token_dim": token_emb.shape[1],
    "svg_scaler": svg_scaler,
    "token_scaler": token_scaler,
    "config": CONFIG,
}
torch.save(checkpoint, dir_output / "dual_ae_checkpoint.pt")

### Eval

In [ ]:
cell_meta_device = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in cell_meta.items()}
# "clusters" is an independent transcriptomics-only clustering already present on
# the raw input adata (not derived from this model's own trans/token projections),
# so it's a valid independent label for cluster-level retrieval credit -- see
# token_retrieval_accuracy's cell_type_labels docstring for why that matters.
cluster_labels = pd.factorize(trans_sub.obs["clusters"])[0]
token_eval_results = token_retrieval_accuracy(
    model, data, token_emb_t, cell_meta_device, device=device,
    cell_type_labels=cluster_labels,
)

### Save latents

In [ ]:
# Reuses token_retrieval_accuracy's own full-population encode instead of a
# second encode_full_graph_batched call.
z_trans_all = token_eval_results["z_trans_cell_full"]
z_token_all = token_eval_results["z_token_cell_full"]

np.savez(
    dir_output / "dual_ae_latents.npz",
    z_trans_cell=z_trans_all,
    z_token_cell=z_token_all,
    patch_row=data.patch_row.numpy(),
    slide_code=data.slide_code.numpy(),
)

trans_sub.obsm["X_dae_trans_cell"] = z_trans_all
trans_sub.obsm["X_dae_token_cell"] = z_token_all
trans_sub.obsm["X_dae_trans_token_concat"] = np.concatenate([z_trans_all, z_token_all], axis=1)

trans_sub.write_h5ad(dir_output / "trans_with_dae_latents.h5ad")
print("Done. Outputs written to:", dir_output)